# Reconstruct the Dozer Dataset

This notebook is the modern replacement walkthrough for the missing private `dozer_dataset.ipynb`. It starts from recovered or freshly generated dozer render/mask capture pairs, composites them onto COCO background images, and writes the dataset layout expected by the historical TensorFlow Object Detection training notebook.

The goal is reproducibility: show how the surviving artifacts become `dataset/train` and `dataset/test`, then generate a larger training dataset for the next notebook.


## 1. Project Setup

Run this notebook from the repository root. In a local checkout, that means the directory containing `README.md`, `scripts/`, and `data/captures/`.

In Google Colab, opening the notebook file by itself is not enough: Colab starts in `/content` and does not automatically include this project repository, the recovered captures, or the COCO background archive. Before continuing in Colab, put the project files under `/content/dozer-detector` or mount/copy them from Drive.

Required inputs:

- `data/captures/{train,test}`: paired `*_img.png` and `*_layer.png` files.
- `data/external/val2017.zip`: local COCO validation images used as random backgrounds.

Generated datasets are ignored by Git because they can be recreated.


In [ ]:
from pathlib import Path
import csv
import random
import subprocess
import sys
import zipfile

import numpy as np
from IPython.display import Image as NotebookImage, display
from PIL import Image, ImageDraw, ImageOps

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_project_root() -> Path:
    candidates = [
        Path.cwd(),
        Path("/content/dozer-detector"),
        Path("/content/drive/MyDrive/dozer-detector"),
    ]
    for candidate in candidates:
        if (candidate / "scripts" / "rebuild_dozer_dataset.py").exists():
            return candidate
    return Path.cwd()


ROOT = find_project_root()
CAPTURES_DIR = ROOT / "data" / "captures"
BACKGROUNDS_ZIP = ROOT / "data" / "external" / "val2017.zip"
SMOKE_DATASET_DIR = ROOT / "dataset_smoke"
TRAINING_DATASET_DIR = ROOT / "dataset"

print("Running in Colab:", IN_COLAB)
print("Project root:", ROOT)
print("Captures dir:", CAPTURES_DIR, CAPTURES_DIR.exists())
print("COCO backgrounds zip:", BACKGROUNDS_ZIP, BACKGROUNDS_ZIP.exists())


In [ ]:
missing = []
if not (ROOT / "scripts" / "rebuild_dozer_dataset.py").exists():
    missing.append(f"project scripts at {ROOT / 'scripts'}")
if not CAPTURES_DIR.exists():
    missing.append(f"captures directory at {CAPTURES_DIR}")
if not BACKGROUNDS_ZIP.exists():
    missing.append(f"COCO backgrounds archive at {BACKGROUNDS_ZIP}")

if missing:
    message = "Missing required project inputs:\n- " + "\n- ".join(missing)
    if IN_COLAB:
        message += """

Colab setup options:

1. Clone or upload this whole project so it exists at /content/dozer-detector.
2. Ensure the recovered captures are at /content/dozer-detector/data/captures.
3. Place the COCO validation archive at /content/dozer-detector/data/external/val2017.zip.

If the project is in Google Drive, mount Drive and either copy it to /content/dozer-detector or update ROOT above to that Drive path.
"""
    raise FileNotFoundError(message)

print("All required inputs found.")


### Optional Colab Input Setup

Use this section only when running in Colab and the preflight cell above says inputs are missing. The notebook needs the project files plus `data/external/val2017.zip`.

If you have the project in Google Drive, mount Drive and set `ROOT` to that folder. If you are starting from a GitHub repo, clone it to `/content/dozer-detector`, then upload or download `val2017.zip` into `data/external/`.


In [ ]:
# Colab-only examples. Uncomment and edit one path/source as needed.

# from google.colab import drive
# drive.mount("/content/drive")
# ROOT = Path("/content/drive/MyDrive/dozer-detector")

# Or, if the project is available on GitHub:
# !git clone YOUR_REPO_URL /content/dozer-detector
# ROOT = Path("/content/dozer-detector")

# After setting ROOT, refresh derived paths:
# CAPTURES_DIR = ROOT / "data" / "captures"
# BACKGROUNDS_ZIP = ROOT / "data" / "external" / "val2017.zip"
# SMOKE_DATASET_DIR = ROOT / "dataset_smoke"
# TRAINING_DATASET_DIR = ROOT / "dataset"

# If val2017.zip is not present, download COCO val2017 directly:
# import urllib.request
# (ROOT / "data" / "external").mkdir(parents=True, exist_ok=True)
# urllib.request.urlretrieve("http://images.cocodataset.org/zips/val2017.zip", BACKGROUNDS_ZIP)


## 2. Inspect Capture Pairs

Each capture has two files:

- `*_img.png`: the rendered dozer foreground.
- `*_layer.png`: an RGB layer image where red pixels identify the dozer.

The reconstruction script thresholds the red layer into a foreground mask.


In [ ]:
sys.path.insert(0, str(ROOT / "scripts"))
from rebuild_dozer_dataset import CapturePair, crop_foreground, find_pairs, load_background, mask_bbox, red_foreground_mask

train_pairs = find_pairs(CAPTURES_DIR, "train")
test_pairs = find_pairs(CAPTURES_DIR, "test")

print(f"Train capture pairs: {len(train_pairs)}")
print(f"Test capture pairs: {len(test_pairs)}")
print("Example train pair:")
print("  image:", train_pairs[0].image.relative_to(ROOT))
print("  layer:", train_pairs[0].layer.relative_to(ROOT))


In [ ]:
def show_side_by_side(images, labels, thumb_size=(320, 320)):
    thumbs = []
    for image in images:
        if isinstance(image, Path):
            image = Image.open(image).convert("RGB")
        else:
            image = image.convert("RGB")
        image.thumbnail(thumb_size, Image.Resampling.LANCZOS)
        thumbs.append(image.copy())

    label_height = 28
    width = sum(img.width for img in thumbs)
    height = max(img.height for img in thumbs) + label_height
    canvas = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(canvas)

    x = 0
    for img, label in zip(thumbs, labels):
        canvas.paste(img, (x, label_height))
        draw.text((x + 8, 8), label, fill="black")
        x += img.width
    display(canvas)

example_pair = train_pairs[0]
show_side_by_side(
    [example_pair.image, example_pair.layer],
    ["capture image", "red layer mask"],
)


## 3. Extract the Foreground Mask

The red layer is converted into a binary mask. The mask gives two things needed by the training dataset:

- the transparent foreground that can be pasted onto a background;
- the bounding box written to `annotations.csv`.


In [ ]:
image = Image.open(example_pair.image).convert("RGB")
layer = Image.open(example_pair.layer).convert("RGB")
mask_array = red_foreground_mask(layer)
xmin, ymin, xmax, ymax = mask_bbox(mask_array)

mask_image = Image.fromarray((mask_array.astype(np.uint8) * 255), mode="L")
boxed_image = image.copy()
draw = ImageDraw.Draw(boxed_image)
draw.rectangle((xmin, ymin, xmax, ymax), outline="red", width=4)

foreground, alpha = crop_foreground(example_pair)

print("Mask bounding box:", {"xmin": xmin, "ymin": ymin, "xmax": xmax, "ymax": ymax})
show_side_by_side(
    [boxed_image, mask_image.convert("RGB"), foreground.convert("RGB")],
    ["render + box", "binary mask", "cropped foreground"],
)


## 4. Composite One Example

The generated dataset is built by placing extracted foregrounds onto random COCO validation images. The full script repeats this process many times with random capture pairs, backgrounds, scales, and positions.


In [ ]:
with zipfile.ZipFile(BACKGROUNDS_ZIP) as zf:
    background_names = [
        name for name in zf.namelist()
        if name.lower().endswith((".jpg", ".jpeg", ".png")) and not name.endswith("/")
    ]

rng = random.Random(20210523)
background = load_background(BACKGROUNDS_ZIP, rng.choice(background_names), size=512)
foreground_rgba, foreground_alpha = crop_foreground(example_pair)
foreground_rgba.thumbnail((360, 360), Image.Resampling.LANCZOS)

canvas = background.convert("RGBA")
x = (canvas.width - foreground_rgba.width) // 2
y = (canvas.height - foreground_rgba.height) // 2
canvas.alpha_composite(foreground_rgba, (x, y))

composite = canvas.convert("RGB")
draw = ImageDraw.Draw(composite)
draw.rectangle((x, y, x + foreground_rgba.width, y + foreground_rgba.height), outline="red", width=3)

show_side_by_side(
    [background, foreground_rgba.convert("RGB"), composite],
    ["COCO background", "foreground", "composite + box"],
)


## 5. Generate a Smoke Dataset

Start with a tiny dataset so failures are quick to diagnose. This writes the same directory shape as the full training dataset, just with fewer images.


In [ ]:
smoke_build = [
    sys.executable,
    "scripts/rebuild_dozer_dataset.py",
    "--output-dir", str(SMOKE_DATASET_DIR),
    "--train-count", "8",
    "--test-count", "4",
    "--preview-count", "8",
]
subprocess.run(smoke_build, cwd=ROOT, check=True)

smoke_validate = [
    sys.executable,
    "scripts/validate_dataset.py",
    str(SMOKE_DATASET_DIR),
    "--expected-train", "8",
    "--expected-test", "4",
]
subprocess.run(smoke_validate, cwd=ROOT, check=True)


In [ ]:
display(NotebookImage(filename=str(SMOKE_DATASET_DIR / "preview.jpg")))


## 6. Inspect the Dataset Shape

`dozer_object_detector` expects generated images, masks, CSV annotations, and a TensorFlow Object Detection label map.


In [ ]:
def count_files(path, pattern):
    return len(list(path.glob(pattern)))

for dataset_dir in [SMOKE_DATASET_DIR]:
    print(dataset_dir.name)
    print("  label map:", (dataset_dir / "label_map.pbtxt").exists())
    print("  train images:", count_files(dataset_dir / "train" / "generated", "*.jpg"))
    print("  train masks:", count_files(dataset_dir / "train" / "mask", "*.png"))
    print("  test images:", count_files(dataset_dir / "test" / "generated", "*.jpg"))
    print("  test masks:", count_files(dataset_dir / "test" / "mask", "*.png"))
    print("  train csv:", dataset_dir / "train" / "annotation" / "annotations.csv")


In [ ]:
annotations_path = SMOKE_DATASET_DIR / "train" / "annotation" / "annotations.csv"
with annotations_path.open(newline="", encoding="utf-8") as fp:
    reader = csv.DictReader(fp)
    rows = list(reader)[:5]

for row in rows:
    print(row)


## 7. Generate the Training Dataset

The smoke dataset proves the pipeline. The training dataset should be larger because each example randomizes the background, placement, and scale. With the recovered capture set, this improves compositing diversity, though it does not fully replace the viewpoint diversity you would get from more Blender captures.

The default training run below creates:

- `10,000` training images
- `2,000` test images
- `dataset/preview.jpg`
- TensorFlow-compatible `annotations.csv` files
- `label_map.pbtxt`

This can take longer than the smoke run. Re-run this cell whenever you want to regenerate the training dataset from the current captures.


In [ ]:
TRAIN_COUNT = 10_000
TEST_COUNT = 2_000
PREVIEW_COUNT = 16

training_build = [
    sys.executable,
    "scripts/rebuild_dozer_dataset.py",
    "--output-dir", str(TRAINING_DATASET_DIR),
    "--train-count", str(TRAIN_COUNT),
    "--test-count", str(TEST_COUNT),
    "--preview-count", str(PREVIEW_COUNT),
]
subprocess.run(training_build, cwd=ROOT, check=True)

training_validate = [
    sys.executable,
    "scripts/validate_dataset.py",
    str(TRAINING_DATASET_DIR),
    "--expected-train", str(TRAIN_COUNT),
    "--expected-test", str(TEST_COUNT),
]
subprocess.run(training_validate, cwd=ROOT, check=True)


In [ ]:
display(NotebookImage(filename=str(TRAINING_DATASET_DIR / "preview.jpg")))


## 8. Final Output for `dozer_object_detector`

The final generated directory should look like this:

```text
dataset/
  label_map.pbtxt
  preview.jpg
  train/
    generated/*.jpg
    mask/*.png
    annotation/annotations.csv
  test/
    generated/*.jpg
    mask/*.png
    annotation/annotations.csv
```

This is the dataset handoff point for the historical TensorFlow Object Detection notebook. The next step is to update that training notebook so it reads this generated `dataset/` directory from the flattened project structure.


In [ ]:
print((TRAINING_DATASET_DIR / "label_map.pbtxt").read_text(encoding="utf-8"))

train_csv = TRAINING_DATASET_DIR / "train" / "annotation" / "annotations.csv"
with train_csv.open(newline="", encoding="utf-8") as fp:
    reader = csv.DictReader(fp)
    first_row = next(reader)

print("First training annotation row:")
print(first_row)


## Notes

- The original private dataset notebook was not recovered.
- This notebook recreates the dataset shape from surviving captures and masks.
- Larger generated datasets improve background and placement variety, but more foreground pose/viewpoint diversity requires new Blender captures.
- YOLO labels are supported by `scripts/rebuild_dozer_dataset.py --write-yolo`, but this notebook intentionally stays focused on the TensorFlow dataset used by `dozer_object_detector`.
